# 🧠 Notebook 02 — Model Inference + Attention Extraction
## VERA: Visual Evidence–Report Alignment

This notebook runs VLM inference on CXR images and extracts cross-modal attention maps.

**Steps:**
1. Load CheXagent-2-3b (or LLaVA-Med) from HuggingFace
2. Test on a single image (attention sanity check)
3. Run batch inference on all images
4. Save generated reports + attention maps

**⚠️ Requires GPU** — Run on Google Colab (T4/A100), Kaggle, or local GPU ≥8GB VRAM.

## 1. Setup & Installation

In [ ]:
# Install dependencies (uncomment for Colab/Kaggle)
# !pip install -q torch torchvision transformers accelerate bitsandbytes
# !pip install -q Pillow tqdm numpy scipy huggingface-hub

import sys
import os
from pathlib import Path
import torch

# Add project root to path (auto-detect Kaggle vs local)
if os.path.exists('/kaggle/working'):
    PROJECT_ROOT = Path('/kaggle/working')
else:
    PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

# Check GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    device = 'cuda'
else:
    print("⚠️ No GPU detected! Inference will be very slow on CPU.")
    device = 'cpu'

In [ ]:
from config import (
    PROCESSED_DIR, ATTENTION_DIR, HF_TOKEN, IS_KAGGLE,
    CHEXAGENT_MODEL_ID, LLAVA_MED_MODEL_ID,
    DEFAULT_MODEL, MAX_NEW_TOKENS, REPORT_PROMPT,
    PATCH_GRID_CHEXAGENT, PATCH_GRID_LLAVA,
    NUM_ATTENTION_LAYERS, FIGURES_DIR,
)
from src.data_utils import load_json, save_json, load_image
from src.attention_extractor import (
    AttentionExtractor, load_chexagent, load_llava_med, process_batch
)

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

## 2. Configuration

Choose which model to run. Start with CheXagent-2-3b (lighter, domain-specific).

In [ ]:
# === MODEL SELECTION ===
# Change this to switch models
USE_MODEL = "chexagent"  # Options: "chexagent" or "llava_med"

# Number of images to process (set to None for full dataset)
# Start with a small number to test the pipeline!
MAX_IMAGES = 50  # Set to None for full dataset

# Use 4-bit quantization to save VRAM?
USE_4BIT = True  # Set True if VRAM < 12GB

if USE_MODEL == "chexagent":
    MODEL_ID = CHEXAGENT_MODEL_ID
    PATCH_GRID = PATCH_GRID_CHEXAGENT
    print(f"Using CheXagent-2-3b ({MODEL_ID})")
else:
    MODEL_ID = LLAVA_MED_MODEL_ID
    PATCH_GRID = PATCH_GRID_LLAVA
    print(f"Using LLaVA-Med ({MODEL_ID})")

print(f"Patch grid: {PATCH_GRID}")
print(f"Max images: {MAX_IMAGES}")
print(f"4-bit quantization: {USE_4BIT}")

## 3. Load Model

In [ ]:
print("="*60)
print(f"Loading model: {MODEL_ID}")
print("="*60)

if USE_MODEL == "chexagent":
    model, tokenizer, image_processor = load_chexagent(
        model_id=MODEL_ID,
        hf_token=HF_TOKEN,
        device=device,
        load_in_4bit=USE_4BIT,
    )
else:
    model, tokenizer, image_processor = load_llava_med(
        model_id=MODEL_ID,
        hf_token=HF_TOKEN,
        device=device,
        load_in_4bit=USE_4BIT,
    )

# Print model info
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params / 1e9:.2f}B")
print(f"Model dtype: {next(model.parameters()).dtype}")

In [ ]:
# Initialize attention extractor
extractor = AttentionExtractor(
    model=model,
    tokenizer=tokenizer,
    image_processor=image_processor,
    patch_grid=PATCH_GRID,
    num_layers_to_use=NUM_ATTENTION_LAYERS,
    device=device,
)

print("✅ AttentionExtractor initialized")

## 4. Single Image Test (Sanity Check)

**Critical:** Test on one image before scaling. Visualize the attention map to verify it looks anatomically sensible.

In [ ]:
# Load dataset
test_data = load_json(str(PROCESSED_DIR / 'test.json'))
print(f"Loaded {len(test_data)} test samples")

# Pick first sample
sample = test_data[0]
print(f"\nImage: {sample['image_id']}")
print(f"Reference findings: {sample['findings'][:200]}")

In [ ]:
# Run single image test
print("Running inference on single image...")
test_image = Image.open(sample['image_path']).convert('RGB')

result = extractor.extract_attention(
    image=test_image,
    prompt=REPORT_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
)

print(f"\n✅ Generated report ({result['num_generated_tokens']} tokens):")
print(f"---")
print(result['generated_text'])
print(f"---")
print(f"\nAttention map shape: {result['attention_maps'].shape}")
print(f"Expected: [num_tokens, {PATCH_GRID[0]}, {PATCH_GRID[1]}]")

In [ ]:
# Visualize attention heatmap
from scipy.ndimage import zoom

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original image
axes[0].imshow(test_image)
axes[0].set_title('Original CXR', fontsize=14)
axes[0].axis('off')

# Average attention over all generated tokens
avg_attention = result['attention_maps'].mean(axis=0)
print(f"Average attention shape: {avg_attention.shape}")

# Upscale to image size
H_img, W_img = np.array(test_image).shape[:2]
attn_upscaled = zoom(avg_attention, (H_img / avg_attention.shape[0], W_img / avg_attention.shape[1]), order=1)

# Attention heatmap
axes[1].imshow(test_image)
axes[1].imshow(attn_upscaled, cmap='jet', alpha=0.5)
axes[1].set_title('Attention Heatmap (all tokens)', fontsize=14)
axes[1].axis('off')

# Raw attention grid
axes[2].imshow(avg_attention, cmap='hot', interpolation='nearest')
axes[2].set_title(f'Raw Attention Grid ({PATCH_GRID[0]}x{PATCH_GRID[1]})', fontsize=14)
axes[2].axis('off')

plt.suptitle(f'Attention Sanity Check — {sample["image_id"]}', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'attention_sanity_check.png'), dpi=150)
plt.show()

print("\n🔍 CHECK: Does the attention look anatomically sensible?")
print("   - Attention should concentrate on lung/cardiac regions")
print("   - If it looks like uniform noise, there may be an extraction issue")

## 5. Batch Inference

Run inference on all images (or a subset). This is the longest step.

Results are checkpointed every 10 images, so you can resume if interrupted.

In [ ]:
# Load all splits for processing
all_data = []
for split in ['train', 'val', 'test']:
    split_data = load_json(str(PROCESSED_DIR / f'{split}.json'))
    for entry in split_data:
        entry['split'] = split
    all_data.extend(split_data)

print(f"Total images to process: {len(all_data)}")

# Limit if MAX_IMAGES is set
if MAX_IMAGES is not None:
    all_data = all_data[:MAX_IMAGES]
    print(f"Limited to {MAX_IMAGES} images for testing")

In [ ]:
print("="*60)
print(f"Running batch inference on {len(all_data)} images")
print("="*60)

output_dir = str(ATTENTION_DIR / USE_MODEL)

results = process_batch(
    extractor=extractor,
    data=all_data,
    output_dir=output_dir,
    prompt=REPORT_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    save_every=10,
)

# Summary
successful = [r for r in results if 'error' not in r]
failed = [r for r in results if 'error' in r]
print(f"\n✅ Successfully processed: {len(successful)}/{len(results)}")
if failed:
    print(f"❌ Failed: {len(failed)}")
    for f in failed[:5]:
        print(f"   - {f['image_id']}: {f['error'][:100]}")

## 6. Save Results & Verify

In [ ]:
# Save inference results with model info
inference_meta = {
    'model_id': MODEL_ID,
    'model_name': USE_MODEL,
    'patch_grid': PATCH_GRID,
    'num_layers_used': NUM_ATTENTION_LAYERS,
    'max_new_tokens': MAX_NEW_TOKENS,
    'num_processed': len(successful),
    'num_failed': len(failed),
    'prompt': REPORT_PROMPT,
}
save_json(inference_meta, str(ATTENTION_DIR / USE_MODEL / 'inference_meta.json'))

# Verify: check a random attention map file
if successful:
    sample_result = successful[0]
    attn_path = sample_result.get('attention_map_path', '')
    if attn_path and os.path.exists(attn_path):
        attn_data = np.load(attn_path)
        print(f"\n--- Verification ---")
        print(f"Attention map file: {attn_path}")
        print(f"Shape: {attn_data['attention_maps'].shape}")
        print(f"Min: {attn_data['attention_maps'].min():.6f}")
        print(f"Max: {attn_data['attention_maps'].max():.6f}")
        print(f"Mean: {attn_data['attention_maps'].mean():.6f}")
        print(f"\n✅ Verification passed!")

print(f"\n📁 Results saved to: {ATTENTION_DIR / USE_MODEL}")
print(f"\nNext: Run 03_claim_extraction.ipynb")